# IND320 – Project work, part 4
**Name:** Kittel Johannes Lande

**Links**
**GitHub repository:** https://github.com/Kitteljl/IND320-app

**Streamlit app:** https://ind320-app-kittel.streamlit.app/

**Panopto Video:** 

**Contents**
1. Setup
2. Currency exchange rates (Norges Bank)
3. Timezone comparison across data sources
4. Forecasting with River
5. Testing the forecast function
6. Streamlit app: structure sketch and design choices
7. Log and AI usage

## 1. Setup
Working directory, imports and shared settings. The notebook runs locally from the project root.

In [1]:
# Run from the project root, even if the notebook lives in notebooks/
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: /Users/kittellande/Documents/Ind320 - Kitteljl


In [2]:
# Imports used throughout the notebook
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

# Project modules from earlier parts
from utils import reservoir_series              # Magasinstatistikk data
from weather import download_weather            # Open-Meteo data
from exchange_rates import download_exchange_rates  # Norges Bank data (new in part 4)
from forecasting import forecast_filling, holdout_eval, rolling_eval

# Shared settings
SEED = 42
RNG = np.random.default_rng(SEED)
pd.options.display.float_format = "{:.4f}".format

2026-11-16 14:33:41.359 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-16 14:33:41.360 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-16 14:33:41.361 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-16 14:33:41.362 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [3]:
# Shared Plotly template so plots match the app theme (same colours as .streamlit/config.toml)
import plotly.io as pio

pio.templates["ind320"] = go.layout.Template(
    layout=dict(
        font=dict(family="sans-serif", size=13),
        colorway=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"],
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
)
pio.templates.default = "plotly+ind320"

## 2. Currency exchange rates (Norges Bank)
`download_exchange_rates(start, end, currencies)` downloads daily EUR and USD rates from Norges Bank's
SDMX API. Rates are quoted as NOK per 1 unit of the foreign currency and exist only on business days.

In [4]:

from exchange_rates import download_exchange_rates
fx = download_exchange_rates("2024-01-01", "2024-12-31")
display(fx.head())

BASE_CUR,NOK_per_EUR,NOK_per_USD
date,,
2024-01-02,11.2815,10.2971
2024-01-03,11.3200,10.3672
2024-01-04,11.2845,10.3027
2024-01-05,11.3090,10.3553
2024-01-08,11.3620,10.3800


## 3. Timezone comparison across data sources
*(filled in after the investigation)*

### 3.1 Open-Meteo
Open-Meteo returns hourly data in the timezone given by the `timezone` parameter (default GMT/UTC).
The test below compares `GMT` and `Europe/Oslo` over the daylight-saving change on 31 March 2024.

In [5]:
# Compare Open-Meteo's two timezone settings around the DST change (31 March 2024)
def om_hourly(tz):
    r = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={"latitude": 59.9, "longitude": 10.7,
                "start_date": "2024-03-30", "end_date": "2024-04-01",
                "hourly": "temperature_2m", "timezone": tz},
        timeout=60,
    )
    r.raise_for_status()
    j = r.json()
    df = pd.DataFrame({"time": pd.to_datetime(j["hourly"]["time"]),
                       "temp": j["hourly"]["temperature_2m"]})
    return j, df

for tz in ["GMT", "Europe/Oslo"]:
    j, df = om_hourly(tz)
    print(f"{tz:12s} -> {j['timezone']} ({j['timezone_abbreviation']}), "
          f"rows: {len(df)}, first: {df.time.iloc[0]}, last: {df.time.iloc[-1]}")
    print("   rows per day:", {str(k): v for k, v in df.groupby(df.time.dt.date).size().items()})

GMT          -> GMT (GMT), rows: 72, first: 2024-03-30 00:00:00, last: 2024-04-01 23:00:00
   rows per day: {'2024-03-30': 24, '2024-03-31': 24, '2024-04-01': 24}
Europe/Oslo  -> Europe/Oslo (GMT+1), rows: 72, first: 2024-03-30 00:00:00, last: 2024-04-01 23:00:00
   rows per day: {'2024-03-30': 24, '2024-03-31': 24, '2024-04-01': 24}


In [6]:
# Compare the text labels with the true UTC instants (unixtime) for Europe/Oslo
def om_both(tz):
    base = {"latitude": 59.9, "longitude": 10.7,
            "start_date": "2024-03-30", "end_date": "2024-04-01",
            "hourly": "temperature_2m", "timezone": tz}
    url = "https://archive-api.open-meteo.com/v1/archive"
    iso = requests.get(url, params=base, timeout=60).json()
    uni = requests.get(url, params={**base, "timeformat": "unixtime"}, timeout=60).json()
    labels = pd.to_datetime(iso["hourly"]["time"])                       # text labels (naive)
    instants = pd.to_datetime(uni["hourly"]["time"], unit="s", utc=True)  # true UTC instants
    return iso, labels, instants

for tz in ["GMT", "Europe/Oslo"]:
    iso, labels, instants = om_both(tz)
    offset = (labels - instants.tz_localize(None)).value_counts()
    print(f"{tz:12s} utc_offset_seconds = {iso['utc_offset_seconds']}")
    print("   first/last true UTC instant:", instants[0], "->", instants[-1])
    print("   label minus UTC (hours):", {str(k): int(v) for k, v in offset.items()})

GMT          utc_offset_seconds = 0
   first/last true UTC instant: 2024-03-30 00:00:00+00:00 -> 2024-04-01 23:00:00+00:00
   label minus UTC (hours): {'0 days 00:00:00': 72}
Europe/Oslo  utc_offset_seconds = 3600
   first/last true UTC instant: 2024-03-29 23:00:00+00:00 -> 2024-04-01 22:00:00+00:00
   label minus UTC (hours): {'0 days 01:00:00': 72}


In [7]:
# Safe approach: request GMT, convert to local time in pandas
_, df_utc = om_hourly("GMT")
df_utc["time_utc"] = df_utc["time"].dt.tz_localize("UTC")
df_utc["time_oslo"] = df_utc["time_utc"].dt.tz_convert("Europe/Oslo")
print("Rows per Oslo day:", df_utc.groupby(df_utc["time_oslo"].dt.date).size().to_dict())
display(df_utc.iloc[28:34][["time_utc", "time_oslo", "temp"]])  # rows around the switch

Rows per Oslo day: {datetime.date(2024, 3, 30): 23, datetime.date(2024, 3, 31): 23, datetime.date(2024, 4, 1): 24, datetime.date(2024, 4, 2): 2}


,time_utc,time_oslo,temp
28,2024-03-31 04:00:00+00:00,2024-03-31 06:00:00+02:00,5.3000
29,2024-03-31 05:00:00+00:00,2024-03-31 07:00:00+02:00,5.3000
30,2024-03-31 06:00:00+00:00,2024-03-31 08:00:00+02:00,5.4000
31,2024-03-31 07:00:00+00:00,2024-03-31 09:00:00+02:00,5.8000
32,2024-03-31 08:00:00+00:00,2024-03-31 10:00:00+02:00,5.9000
33,2024-03-31 09:00:00+00:00,2024-03-31 11:00:00+02:00,6.4000


In [8]:
display(df_utc.iloc[23:28][["time_utc", "time_oslo", "temp"]])
print("Rows per Oslo day:", df_utc.groupby(df_utc["time_oslo"].dt.date).size().to_dict())

,time_utc,time_oslo,temp
23,2024-03-30 23:00:00+00:00,2024-03-31 00:00:00+01:00,5.4000
24,2024-03-31 00:00:00+00:00,2024-03-31 01:00:00+01:00,5.2000
25,2024-03-31 01:00:00+00:00,2024-03-31 03:00:00+02:00,5.5000
26,2024-03-31 02:00:00+00:00,2024-03-31 04:00:00+02:00,5.5000
27,2024-03-31 03:00:00+00:00,2024-03-31 05:00:00+02:00,5.3000


Rows per Oslo day: {datetime.date(2024, 3, 30): 23, datetime.date(2024, 3, 31): 23, datetime.date(2024, 4, 1): 24, datetime.date(2024, 4, 2): 2}


**Finding (Open-Meteo):** With `timezone=GMT` the hourly labels equal UTC. With `timezone=Europe/Oslo`
the API applied one fixed offset (+1 h) to the whole 3-day response, so the labels after the DST change
on 31 March 2024 were one hour wrong and the 23-hour local day was returned as 24 rows.
**Solution:** download in GMT and convert with `tz_convert("Europe/Oslo")` in pandas.

### 3.2 Norges Bank

In [9]:
# NB publishes one rate per business day (no time of day, no weekends, no holidays).
# Easter 2024 (28 Mar, 29 Mar, 1 Apr) is a good test of the gaps.
fx_easter = download_exchange_rates("2024-03-25", "2024-04-05")
full_days = pd.date_range("2024-03-25", "2024-04-05", freq="D")

print("Weekday numbers present (0=Mon ... 6=Sun):", sorted(fx_easter.index.dayofweek.unique()))
print("Dates WITHOUT a rate:", [d.strftime("%a %Y-%m-%d") for d in full_days.difference(fx_easter.index)])

# One solution: forward-fill to a calendar-day index so it can be joined with other sources
fx_daily = fx_easter.reindex(full_days).ffill()
display(fx_daily)

Weekday numbers present (0=Mon ... 6=Sun): [0, 1, 2, 3, 4]
Dates WITHOUT a rate: ['Thu 2024-03-28', 'Fri 2024-03-29', 'Sat 2024-03-30', 'Sun 2024-03-31', 'Mon 2024-04-01']


/var/folders/kz/zz1m13255tzdllktgg86q5680000gn/T/ipykernel_58396/1008861989.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  print("Weekday numbers present (0=Mon ... 6=Sun):", sorted(fx_easter.index.dayofweek.unique()))


BASE_CUR,NOK_per_EUR,NOK_per_USD
2024-03-25,11.6050,10.7107
2024-03-26,11.6260,10.7103
2024-03-27,11.6825,10.8011
2024-03-28,11.6825,10.8011
2024-03-29,11.6825,10.8011
2024-03-30,11.6825,10.8011
2024-03-31,11.6825,10.8011
2024-04-01,11.6825,10.8011
2024-04-02,11.7080,10.8922
2024-04-03,11.6580,10.8115


**Finding (Norges Bank):** The exchange rates are one value per business day, with a date but no time of day.
There are no rates on weekends or Norwegian bank holidays (for example 28 Mar, 29 Mar and 1 Apr 2024 at Easter).
The rate is a daily reference rate and has no timezone, so it cannot be matched by the hour.
**Solution:** Reindex to calendar days and forward-fill (`ffill`), then join on the date. Each hour or week
uses the rate of the same day. On weekends and holidays, where no rate exists, the most recent published
rate is used. Forward-filling only uses past values, so no future information leaks into the forecasts.

### 3.3 Magasinstatistikk (NVE)

In [10]:
# Raw API response: how are the dates defined?
r = requests.get(
    "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData",
    timeout=60,
)
r.raise_for_status()
mag = pd.DataFrame(r.json())
print(list(mag.columns))
display(mag.head(3))

['dato_Id', 'omrType', 'omrnr', 'iso_aar', 'iso_uke', 'fyllingsgrad', 'kapasitet_TWh', 'fylling_TWh', 'neste_Publiseringsdato', 'fyllingsgrad_forrige_uke', 'endring_fyllingsgrad']


,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,2020-11-15,EL,1,2020,46,0.9744,6.0033,5.8493,2020-11-25T13:00:00,0.9974,-0.0230
1,2011-08-28,EL,4,2011,34,0.7865,21.0792,16.5788,0001-01-01T00:00:00,0.7872,-0.0007
2,1998-07-05,EL,3,1998,27,0.7940,8.9209,7.0829,0001-01-01T00:00:00,0.7338,0.0602


In [11]:
mag["date"] = pd.to_datetime(mag["dato_Id"])
iso = mag["date"].dt.isocalendar()

print("Weekday numbers (0=Mon ... 6=Sun):", mag["date"].dt.dayofweek.value_counts().to_dict())
print("Share where ISO week/year match iso_uke/iso_aar:",
      round(((iso.week == mag["iso_uke"]) & (iso.year == mag["iso_aar"])).mean(), 4))
print("Date range:", mag["date"].min().date(), "->", mag["date"].max().date())

# Spacing between observations in one series (price area 1)
one = mag[(mag["omrType"] == "EL") & (mag["omrnr"] == 1)].sort_values("date")
print("Days between observations:", one["date"].diff().dt.days.value_counts().head().to_dict())

# Publication timestamps: weekday and time (0001-01-01 means "not set")
pub = pd.to_datetime(mag["neste_Publiseringsdato"])
pub = pub[pub.dt.year > 1]
print("Publication weekday:", pub.dt.dayofweek.value_counts().to_dict(),
      "| hours:", pub.dt.hour.value_counts().to_dict())

Weekday numbers (0=Mon ... 6=Sun): {6: 14958}
Share where ISO week/year match iso_uke/iso_aar: 1.0
Date range: 1995-01-08 -> 2026-11-08
Days between observations: {7.0: 1661}
Publication weekday: {2: 3591, 3: 36, 4: 9} | hours: {13: 3636}


/var/folders/kz/zz1m13255tzdllktgg86q5680000gn/T/ipykernel_58396/2228224159.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  print("Weekday numbers (0=Mon ... 6=Sun):", mag["date"].dt.dayofweek.value_counts().to_dict())
/var/folders/kz/zz1m13255tzdllktgg86q5680000gn/T/ipykernel_58396/2228224159.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  print("Publication weekday:", pub.dt.dayofweek.value_counts().to_dict(),


**Finding (Magasinstatistikk):** The data is weekly. `dato_Id` is always a Sunday (all 14,958 rows) and equals the
last day of the ISO week given by `iso_aar`/`iso_uke` (100 % match). Observations are exactly 7 days apart,
and there are no time-of-day values or timezone information. New data is published on Wednesdays at 13:00
(local time assumed, since no timezone is given), so the newest value is always several days old.
**Solution:** Aggregate hourly and daily data (weather, prices) to ISO weeks ending on Sunday, using Oslo local
dates, before joining. In pandas: `resample("W-SUN")`.

### 3.4 ENTSO-E (cross-border flows)

**Finding:** The physical flows are hourly mean values stored in MongoDB as timestamps without timezone
information (`datetime64`). A check against the daylight-saving change on 30 March 2025 showed that the
hour 02:00 exists for all 18 connection/direction series. That hour does not exist in Norwegian local time,
so the timestamps are UTC (MongoDB stores dates in UTC). The data covers 2024-01-01 00:00 to
2025-12-31 22:00.
**Solution:** Treat the timestamps as UTC (`tz_localize("UTC")`), which matches Open-Meteo requested
with `timezone=GMT`, and convert to Oslo time (`tz_convert("Europe/Oslo")`) before aggregating to Norwegian
calendar days or weeks for the exchange rates and the NVE data.

### 3.5 Summary

| Source | Time resolution | Timestamp definition | Timezone | Handling |
|---|---|---|---|---|
| Open-Meteo | hourly | label per hour | GMT by default. `Europe/Oslo` gave a fixed +1 h offset, wrong after the DST change | request GMT, convert in pandas |
| ENTSO-E (MongoDB) | hourly | start of the hour, no timezone stored | UTC (checked with the DST hour on 30 Mar 2025) | `tz_localize("UTC")`, convert to Oslo for daily or weekly joins |
| Norges Bank | daily, business days | date only | none (Norwegian reference rate) | reindex to calendar days, `ffill`, join on the Oslo date |
| Magasinstatistikk | weekly | Sunday ending the ISO week | none (publication assumed local) | aggregate to `W-SUN` before joining |

All hourly sources are therefore in UTC. Daily and weekly sources are matched through the Oslo calendar date.

## 4. Forecasting with River

`forecast_filling()` (in `forecasting.py`) predicts the weekly degree of filling of one reservoir region
with River's `SNARIMAX`, so the model learns online, one observation at a time.

**Inputs (all selectable):**
- `y`: the weekly filling series of one region (Magasinstatistikk)
- `history_weeks`: how many of the most recent weeks the model learns from
- `horizon`: number of weeks to forecast
- `p`, `d`, `q`: autoregressive order, differencing and moving-average order
- `fourier_k`: number of yearly harmonics used as exogenous features (0 = none)
- `lr`: learning rate of the underlying linear regressor

**Design choices:**
- **Time resolution:** weekly, the native resolution of Magasinstatistikk. Hourly or daily data would add
  many rows without adding information, since the filling changes slowly.
- **Seasonality:** instead of a seasonal period of `m=52`, which adds many lag terms and is slow, the yearly
  cycle is given as sine and cosine features of the day of year. These are known for future dates, so they
  can be used directly in the forecast.
- **Forecasting:** River's `forecast(horizon, xs)` predicts recursively, using each predicted value as input
  for the next step (dynamic forecasting).
- **Evaluation:** `holdout_eval()` hides the last `horizon` weeks, forecasts them and compares the error (MAE)
  with a naive forecast that repeats the last known value.

In [12]:
# Weekly filling degree for price area 1 (EL 1), as a Series indexed by Sunday dates
y = (mag[(mag["omrType"] == "EL") & (mag["omrnr"] == 1)]
        .set_index("date")["fyllingsgrad"].sort_index())
print(len(y), "weeks, last date:", y.index[-1].date())

1662 weeks, last date: 2026-11-08


In [13]:
fc, actual, mae, naive_mae = holdout_eval(y, horizon=12, history_weeks=260)
print(f"MAE model: {mae:.4f} | MAE naive (last value): {naive_mae:.4f}")

fig = go.Figure()
fig.add_scatter(x=y.index[-80:], y=y.iloc[-80:], name="Observed")
fig.add_scatter(x=fc.index, y=fc.values, name="Forecast", line=dict(dash="dash"))
fig.update_layout(title="12-week hold-out forecast, price area 1", yaxis_title="Filling degree")
fig

MAE model: 0.1183 | MAE naive (last value): 0.0332


## 5. Testing the forecast function



In [14]:
import itertools, time

grid = {
    "history_weeks": [104, 260, 520],
    "p": [1, 2, 4],
    "d": [0, 1],
    "fourier_k": [0, 2],
    "lr": [0.01, 0.1],
}

rows, t0 = [], time.time()
for hw, p, d, k, lr in itertools.product(*grid.values()):
    mae, naive = rolling_eval(y, horizon=12, n_origins=6, step=26,
                              history_weeks=hw, p=p, d=d, q=1, fourier_k=k, lr=lr)
    rows.append(dict(history_weeks=hw, p=p, d=d, fourier_k=k, lr=lr, mae=mae, naive_mae=naive))
print(f"Done in {time.time() - t0:.0f} s")

results = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
display(results.head(10))
print("Naive MAE (same test periods):", round(results["naive_mae"].iloc[0], 4))

Done in 1 s


,history_weeks,p,d,fourier_k,lr,mae,naive_mae
0,104,1,1,2,0.0100,0.0359,0.0960
1,104,4,1,2,0.0100,0.0360,0.0960
2,104,2,1,2,0.0100,0.0367,0.0960
3,520,1,0,2,0.0100,0.0424,0.0960
4,520,2,0,2,0.0100,0.0463,0.0960
5,520,4,0,2,0.0100,0.0610,0.0960
6,520,1,1,2,0.0100,0.0625,0.0960
7,520,2,1,2,0.0100,0.0682,0.0960
8,520,4,1,2,0.0100,0.0767,0.0960
9,520,2,1,0,0.0100,0.0794,0.0960


Naive MAE (same test periods): 0.096


In [15]:
# Validate on different forecast start dates: drop the last 13 weeks so the 6 origins shift
best = dict(history_weeks=104, p=1, d=1, q=1, fourier_k=2, lr=0.01)
mae_v, naive_v = rolling_eval(y.iloc[:-13], horizon=12, n_origins=6, step=26, **best)
print(f"Validation MAE model: {mae_v:.4f} | naive: {naive_v:.4f}")

Validation MAE model: 0.1481 | naive: 0.2463


In [16]:
# Per-origin errors on the validation periods
best = dict(history_weeks=104, p=1, d=1, q=1, fourier_k=2, lr=0.01)
yv = y.iloc[:-13]
rows = []
for k in range(6):
    cut = len(yv) - 12 - k * 26
    train, test = yv.iloc[:cut], yv.iloc[cut:cut + 12]
    fc = forecast_filling(train, horizon=12, **best)
    rows.append(dict(forecast_start=test.index[0].date(),
                     model_mae=round(float(np.mean(np.abs(fc.values - test.values))), 3),
                     naive_mae=round(float(np.mean(np.abs(train.iloc[-1] - test.values))), 3)))
display(pd.DataFrame(rows))

,forecast_start,model_mae,naive_mae
0,2026-05-24,0.2690,0.3550
1,2025-11-23,0.1110,0.2160
2,2025-05-25,0.0740,0.1930
3,2024-11-24,0.0920,0.2010
4,2024-05-26,0.3230,0.2840
5,2023-11-26,0.0210,0.2280


**Test method.** The model was tested with a rolling-origin evaluation on weekly filling data for price area 1
(`EL`, area number 1). For each forecast start date, the model learned only from the history before that date,
forecast the next 12 weeks and was compared with the actual values using the mean absolute error (MAE, in filling
degree, where 1.0 = full). A naive forecast that repeats the last known value served as a baseline.
Six start dates spaced 26 weeks apart (three years, covering all seasons) were used for each test.

**Settings compared.** 72 combinations of: history length (104, 260 or 520 weeks), AR order `p` (1, 2 or 4),
differencing `d` (0 or 1), yearly Fourier terms `fourier_k` (0 or 2) and learning rate (0.01 or 0.1).
The whole grid ran in about X seconds, so the model is fast enough for interactive use in Streamlit.

**Results on the tuning periods.**

| Rank | History (weeks) | p | d | Fourier terms | Learning rate | MAE | Naive MAE |
|---|---|---|---|---|---|---|---|
| 1 | 104 | 1 | 1 | 2 | 0.01 | 0.036 | 0.096 |
| 2 | 104 | 4 | 1 | 2 | 0.01 | 0.036 | 0.096 |
| 3 | 104 | 2 | 1 | 2 | 0.01 | 0.037 | 0.096 |
| 4 | 520 | 1 | 0 | 2 | 0.01 | 0.042 | 0.096 |

- Differencing (`d=1`) and the yearly Fourier terms gave most of the improvement.
- `p=1`, `2` and `4` performed almost the same, so the simplest model (`p=1`) is preferred.
- A short history (104 weeks) performed better than a long one (520 weeks), which suggests that older
  years are less representative of the present.

**Validation on different start dates.** Because the settings were chosen on the same six periods they were
scored on, the result is optimistic. A separate validation set (the data shifted by 13 weeks, giving six new start
dates) gave a model MAE of 0.148 against 0.246 for the naive forecast: still about 40 % better than the baseline,
but four times the error of the tuning score. The realistic accuracy is therefore about 15 percentage points of
filling over a 12-week horizon.

| Forecast start | Model MAE | Naive MAE |
|---|---|---|
| 2026-05-24 | 0.269 | 0.355 |
| 2025-11-23 | 0.111 | 0.216 |
| 2025-05-25 | 0.074 | 0.193 |
| 2024-11-24 | 0.092 | 0.201 |
| 2024-05-26 | 0.323 | 0.284 |
| 2023-11-26 | 0.021 | 0.228 |

**Variation across the year.** The model beat the naive forecast in five of the six periods. Forecasts starting
in late November were accurate (MAE 0.02 to 0.11), while forecasts starting in late May, when the snowmelt filling
begins, were much worse (MAE 0.07 to 0.32), and in 2024 the model was slightly worse than the naive forecast.
The yearly Fourier pattern cannot capture how fast the reservoirs fill in a given year, which probably depends on
snowpack and weather. Weather data is therefore tested as an additional input in the next step.

**A single window can mislead.** On the last 12 weeks alone, the naive forecast looked better (MAE 0.033 against
0.114 for the model), because the filling was almost flat. Averaging over several start dates avoids this.

**Default setting for Streamlit:** `history_weeks=104`, `p=1`, `d=1`, `q=1`, `fourier_k=2`, `lr=0.01`.

In [17]:
import os, pickle, time

os.makedirs("cache", exist_ok=True)     # add "cache/" to .gitignore so it isn't committed
cache_file = "cache/weather_NO1_2021_2026.pkl"

if os.path.exists(cache_file):
    wx = pickle.load(open(cache_file, "rb"))
else:
    t0 = time.time()
    parts = []
    for year in range(2021, 2027):
        end = f"{year}-12-31" if year < 2026 else "2026-11-08"
        parts.append(download_weather("NO1", f"{year}-01-01", end, n_points=5,
                                      hourly=["temperature_2m", "precipitation"]))
        print(year, "done", round(time.time() - t0), "s")
    wx = pd.concat(parts)
    pickle.dump(wx, open(cache_file, "wb"))

print(wx.shape, wx.index.min(), "->", wx.index.max())
display(wx.head(3))

(51312, 2) 2021-01-01 00:00:00 -> 2026-11-08 23:00:00


,temperature_2m,precipitation
time,,
2021-01-01 00:00:00,-2.9000,0.1400
2021-01-01 01:00:00,-3.0200,0.1400
2021-01-01 02:00:00,-3.0800,0.1000


### 5.1 Weather as input

In [18]:
# Weekly weather that matches NVE's Sunday dates:
# UTC hours -> Oslo local days -> weeks ending on Sunday (temperature = mean, precipitation = sum)
local = wx.copy()
local.index = pd.to_datetime(local.index).tz_localize("UTC").tz_convert("Europe/Oslo")
daily = local.resample("D").agg({"temperature_2m": "mean", "precipitation": "sum"})
weekly = daily.resample("W-SUN").agg({"temperature_2m": "mean", "precipitation": "sum"})
weekly.index = weekly.index.tz_localize(None).normalize()
weekly = weekly.iloc[1:]          # the first week (1 Jan 2021) is incomplete
weekly = weekly.loc[:y.index.max()]   # keep only weeks up to the last NVE date (drops the 1-hour "week" at the end)
print(weekly.index.min().date(), "->", weekly.index.max().date(), "| weeks:", len(weekly))
display(weekly.tail(3))

2021-01-10 -> 2026-11-08 | weeks: 305


,temperature_2m,precipitation
time,,
2026-10-25,4.6094,73.7400
2026-11-01,5.7345,12.3200
2026-11-08,-0.6587,0.0000


In [19]:
# Fair comparison: same settings, with and without weather, on both sets of start dates
best = dict(history_weeks=104, p=1, d=1, q=1, fourier_k=2, lr=0.01)
options = {
    "none": None,
    "temperature": weekly[["temperature_2m"]],
    "precipitation": weekly[["precipitation"]],
    "temperature + precipitation": weekly,
}
rows = []
for name, ex in options.items():
    m_val, n_val = rolling_eval(y.iloc[:-13], horizon=12, n_origins=6, step=26, exog=ex, **best)
    m_tun, n_tun = rolling_eval(y, horizon=12, n_origins=6, step=26, exog=ex, **best)
    rows.append(dict(weather=name, mae_validation=round(m_val, 4), mae_tuning=round(m_tun, 4)))
display(pd.DataFrame(rows))
print("Naive (validation / tuning):", round(n_val, 4), "/", round(n_tun, 4))

,weather,mae_validation,mae_tuning
0,none,0.1481,0.0359
1,temperature,0.1088,0.0406
2,precipitation,0.1457,0.0450
3,temperature + precipitation,0.1203,0.0462


Naive (validation / tuning): 0.2463 / 0.096


### 5.1 Weather as input

**Setup.** Hourly ERA5 temperature and precipitation for NO1 (5 sampled points averaged, 2021–2026) were converted to
Oslo local time and aggregated to weeks ending on Sunday, so they match NVE's dates (temperature = weekly mean,
precipitation = weekly sum). The model receives the real weather for the history. For the forecast weeks, the
real weather is unknown, so the average for that week of the year, calculated only from past data, is used. The function
never sees weather after the forecast start date, so there is no leakage from the future.

**Results** (best settings from section 5, MAE for 12 weeks ahead):

| Weather input | Validation MAE | Tuning MAE | Average |
|---|---|---|---|
| none | 0.148 | 0.036 | 0.092 |
| temperature | 0.109 | 0.041 | 0.075 |
| precipitation | 0.146 | 0.045 | 0.095 |
| temperature + precipitation | 0.120 | 0.046 | 0.083 |
| naive baseline | 0.246 | 0.096 | 0.171 |

**Findings.**
- Temperature reduced the error on the validation set by about 27 % but increased it slightly on the tuning set,
  where the settings had been chosen without weather. Averaged over all 12 start dates, temperature was best.
- Precipitation gave no improvement, and adding it to temperature was worse than temperature alone.
- With six start dates per set, the differences are small compared with the variation between seasons, so the
  conclusion is suggestive, not conclusive.

**Default for Streamlit:** `history_weeks=104`, `p=1`, `d=1`, `q=1`, `fourier_k=2`, `lr=0.01`, with temperature as
exogenous input. Precipitation and the other data sources remain selectable.

**Instability at high learning rates.** Learning rates above 0.01 made the online model unstable: because the
forecast is dynamic (each prediction is fed back as input), the error grew from week to week and the forecast
left the possible range of 0-100 %. The app therefore only offers learning rates up to 0.01 and warns, and clips
the values, if a forecast leaves the range.

## 6. Streamlit app: structure sketch and design choices

I planned the structure on paper before changing any code. Two sketches follow: how the pages are grouped
and how they are connected, and how each new or changed page is laid out.

### 6.1 Navigation and structure

![App structure sketch](../images/app_sketch.png)

**Grouping.** The pages are ordered by the type of task:
- **Explore:** the map, the data overview and the cross-border flows.
- **Analyse:** the seasonal analysis (STL and spectrogram) and the outliers (SATV and LOF).
- **Predict:** the sliding window correlation and the forecast.

A front page links to the three groups, and the sidebar menu shows the same groups with headers, so the user
can always see which group is active. The old LineChartColumn page was merged into the data overview page,
because both show an overview of the same data and did not need separate pages.

**Shared selections.** The map is the main place to choose an area and a time. Clicking an area marks it with
a thicker outline and stores the choice in `st.session_state`. The pages in the Analyse and Predict groups read
the same selection by default, so the user does not have to choose the area again on every page. Choices that
only matter for one page, such as the lag, the window length or the model orders, are local to that page.

### 6.2 Page layouts

![Page layouts](../images/page_layouts.png)

Blue = graphics, green = table, yellow = controls, orange = symbol or status, grey = sidebar menu.
All pages follow the same pattern: controls at the top (or in a left column on the forecast page), the main
graphic in the middle, a small summary graphic or metric beside it, and a table at the bottom. This makes
the pages easy to recognise when moving between them.

- **Map:** colour selector (price area, water area or weather property), a time slider, the GeoJSON map with
  click selection, a summary chart for the selected area and a table of values per area.
- **Outliers:** weather property selector (temperature by default), tabs for SATV and LOF, the time series with
  the outliers marked and a table of the outlier dates.
- **Sliding window correlation:** weather property, lag and window length sliders, the correlation over time and a
  scatter plot of weather against filling.
- **Forecast:** model settings, history, horizon and weather input, the animated Plotly forecast, a status area
  with spinner and error messages, and a table of the forecast values.

### 6.3 Design choices

**Theme.** *(colours, fonts and the shared Plotly template, filled in when the theme is chosen)*

**Plots.** All static matplotlib

### 6.4 GeoJSON for the map

In [20]:
import glob
import geopandas as gpd

bud_path = glob.glob("data/**/Bud*.shp", recursive=True)[0]
reg_path = glob.glob("data/**/NVE_Region*.shp", recursive=True)[0]
print(bud_path, "|", reg_path)

bud = gpd.read_file(bud_path, encoding="utf-8")
reg = gpd.read_file(reg_path, encoding="utf-8")

print("Price areas:", bud.crs, list(bud.columns))
display(bud.drop(columns="geometry").head(10))

print("NVE regions:", reg.crs, list(reg.columns))
display(reg.drop(columns="geometry").head(10))

data/Budområder.shp | data/NVE_Region.shp
Price areas: EPSG:25833 ['objectid', 'budomr', 'globalid', 'st_area(sh', 'st_perimet', 'geometry']


,objectid,budomr,globalid,st_area(sh,st_perimet
0,1.0000,NO 1,NaN,60529018620.5849,1657761.1483
1,2.0000,NO 2,NaN,68759689039.3490,1256473.0668
2,3.0000,NO 3,NaN,104219851962.2653,1773322.6269
3,4.0000,NO 4,NaN,203250012051.7293,3385499.2410
4,5.0000,NO 5,NaN,33348702813.9965,1106762.3423


NVE regions: EPSG:25833 ['OBJECTID', 'REGION', 'REGIONNAVN', 'POSTNUMMER', 'POSTSTED', 'BESØKSADR', 'ADRESSE', 'geometry']


,OBJECTID,REGION,REGIONNAVN,POSTNUMMER,POSTSTED,BESØKSADR,ADRESSE
0,1.0000,RV,Region Vest,6801,Førde,Naustdalsveien 1B,Postboks 53
1,2.0000,RM,Region Midt-Norge,7030,Trondheim,"Abelsgate 9, Trondheim",Abelsgate 9
2,3.0000,RN,Region Nord,8514,Narvik,Kongensgate 52-54,Kongensgate 52-54
3,4.0000,RØ,Region Øst,2307,Hamar,Vangsvegen 73,Postboks 4223
4,5.0000,RS,Region Sør,3103,Tønsberg,Anton Jenssens gate 7,Postboks 2124


In [21]:
import os
import geopandas as gpd

os.makedirs("data/geojson", exist_ok=True)

def save_geojson(gdf, path):
    """Simplify (metres, while still in UTM), convert to WGS84 and save as GeoJSON."""
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.simplify(500, preserve_topology=True)
    gdf = gdf.to_crs(4326)
    gdf.to_file(path, driver="GeoJSON")
    print(f"{path}: {len(gdf)} features, {os.path.getsize(path) / 1024:.0f} KB")
    return gdf

# 1) Price areas (omrType EL): "NO 1" ... "NO 5"
el = bud[["budomr", "geometry"]].copy()
el["omrnr"] = el["budomr"].str.extract(r"(\d+)")[0].astype(int)
el["omrType"] = "EL"
el["key"] = "EL" + el["omrnr"].astype(str)
el["name"] = "Price area NO" + el["omrnr"].astype(str)
el = el[["key", "omrType", "omrnr", "name", "geometry"]]

# 2) Water areas (omrType VASS): RS + RØ -> 1, RV -> 2, RM + RN -> 3
vass_map = {"RS": 1, "RØ": 1, "RV": 2, "RM": 3, "RN": 3}
vass = reg[["REGION", "geometry"]].copy()
vass["omrnr"] = vass["REGION"].map(vass_map)
assert vass["omrnr"].notna().all(), "A region code is missing in vass_map"
vass = vass.dissolve(by="omrnr").reset_index()
vass["omrType"] = "VASS"
vass["key"] = "VASS" + vass["omrnr"].astype(str)
vass["name"] = "Water area " + vass["omrnr"].astype(str)
vass = vass[["key", "omrType", "omrnr", "name", "geometry"]]

# 3) Whole Norway (omrType NO, number 0)
norway = gpd.GeoDataFrame(
    {"key": ["NO0"], "omrType": ["NO"], "omrnr": [0], "name": ["Norway"]},
    geometry=[reg.union_all()], crs=reg.crs,
)

for gdf, name in [(el, "el"), (vass, "vass"), (norway, "no")]:
    save_geojson(gdf, f"data/geojson/{name}.geojson")

data/geojson/el.geojson: 5 features, 53 KB
data/geojson/vass.geojson: 3 features, 75 KB
data/geojson/no.geojson: 1 features, 47 KB


## 7. Log and AI usage
## 7. Log and AI usage

### 7.1 AI usage

I used Claude as a step-by-step assistant. It explained the Norges Bank, Open-Meteo and River APIs, wrote
first versions of the functions and Streamlit pages, and helped me debug errors (for example the
timezone check, the unstable learning rate and the navigation setup). I ran and tested every step myself,
chose the design (the blue theme, the page grouping, the sketches and the bonus task), and checked the results
against the data. No passwords or secrets were shared with the repository or placed in the notebook.

### 7.2 Log

In Part 4 I added exchange rates, a timezone comparison and a River forecast to the notebook, and rebuilt the Streamlit app around a map and a prediction section.

**Data and timezones.** `download_exchange_rates()` fetches daily EUR and USD rates from Norges Bank's SDMX API. The rates exist only on business days, so I reindex to calendar days and forward-fill, and each day uses the same-day rate. Comparing the timestamps of the four sources gave the most useful finding: Open-Meteo with `timezone=Europe/Oslo` applies one fixed +1 h offset to the whole response, so the hours after the daylight-saving change were one hour wrong. I therefore request GMT and convert in pandas. The ENTSO-E data in MongoDB is UTC (the hour 02:00 on 30 March 2025 exists), Magasinstatistikk is weekly on Sundays, and Norges Bank has dates only.

**Forecast.** `forecast_filling()` uses River's SNARIMAX on weekly filling with yearly Fourier terms and optional weekly weather as exogenous input. History length, horizon, p, d, q, Fourier terms, learning rate and data sources are arguments. Weekly resolution keeps it fast. I tested 72 settings with a rolling-origin evaluation (six start dates over three years) against a naive forecast. The best setting (104 weeks, p=1, d=1, two Fourier terms, learning rate 0.01) had an MAE of 0.036 against 0.096, but on a separate validation set it was 0.148 against 0.246, so the first score was optimistic. Temperature as input gave the best average. Learning rates above 0.01 made the dynamic forecast unstable, so the app only offers lower values and warns if a forecast leaves 0-100 %.

**Streamlit.** I chose a blue rain theme and grouped the pages as Explore, Analyse and Predict with `st.navigation`, planned with the sketches in section 6. All plots are now Plotly. The new map overlays GeoJSON areas coloured by filling or a weather property for a chosen week; a click outlines the area and stores it in session state, and the other pages start from it. The outlier and anomaly plots let the user choose the weather property (temperature by default). The sliding-window correlation page has selectable lag, window and property, and the forecast page animates the forecast with Plotly and starts from the map's week.

**Correlation findings.** For EL1, temperature and the filling had an overall correlation of 0.64 (R² about 0.42), and the sliding-window correlation varied between about 0.5 and 0.8, so the relationship is fairly strong but not constant. For temperature and precipitation the strongest lag was 0 weeks, but precipitation was almost uncorrelated (0.04 at a 2-week lag) and negative at 8 weeks. Snowfall was negative at short lags (about -0.3), rose with the lag and was strongest at 19 weeks, which fits snow accumulating in winter and filling the reservoirs when it melts months later. Part of this is shared seasonality.

**Bonus: waiting time.** Every slow step is cached and shows a spinner, a progress bar or a status box, and the analysis functions are cached too.